# Week 5 In-Class Exercise: Support Vector Machines on Penguins

In the textbook (Chapter 5), we studied **Support Vector Machines**: large-margin classification with `LinearSVC`, why SVMs are **sensitive to feature scale**, the **soft-margin** trade-off controlled by `C`, and **nonlinear** classification with **polynomial** and **RBF (Gaussian) kernels** — mostly demonstrated on the **iris** dataset and the two-moons toy set.

In this exercise, you'll apply those **same concepts** to a different but structurally similar dataset:

> **Palmer Penguins** — 344 penguins from three species (Adelie, Chinstrap, Gentoo) described by four body measurements: culmen (bill) length and depth, flipper length, and body mass.

Just like iris uses flower measurements to separate three species, penguins use body measurements to separate three species — so every technique from the textbook notebook transfers directly. Your job is to adapt the code, not invent new methods.

You'll follow the same workflow as Chapter 5:
1. Load and explore the data
2. Train a **linear SVM** classifier ("is it a Gentoo?")
3. See why **feature scaling** matters so much for SVMs
4. Explore the **soft-margin** trade-off by varying `C`
5. Use **polynomial** and **RBF kernels** for nonlinear problems
6. Visualize **decision boundaries** and build a **multiclass** SVM

**Data source:** [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/) (Gorman, Williams & Fraser, 2014), loaded from OpenML.

### Useful API References

| Task | Documentation |
|------|---------------|
| Load data | [sklearn.datasets.fetch_openml](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_openml.html) |
| Linear SVM | [sklearn.svm.LinearSVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html) |
| Kernel SVM | [sklearn.svm.SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) |
| Scaling | [sklearn.preprocessing.StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) |
| Pipelines | [sklearn.pipeline.make_pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html) |
| Polynomial features | [sklearn.preprocessing.PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) |
| Two-moons toy data | [sklearn.datasets.make_moons](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) |
| Train/test split | [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) |
| Cross-validation | [cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) |
| Accuracy | [accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html) |

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/pjmcswee/IST707-Notebooks/blob/main/week5/week5_inclass_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

## Setup

Run this cell to import the libraries and set the plotting defaults.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=12)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

## Step 1: Load the Data

The textbook loads iris with `load_iris`. We load Palmer Penguins from OpenML. A handful of rows have missing measurements, so we drop them (that leaves 334 clean penguins). This cell is provided — just run it.

In [ ]:
from sklearn.datasets import fetch_openml

penguins = fetch_openml(name='penguins', version=1, as_frame=True, parser='auto')
df = penguins.frame.dropna().reset_index(drop=True)

feature_cols = ['culmen_length_mm', 'culmen_depth_mm',
                'flipper_length_mm', 'body_mass_g']

print('Shape (after dropping missing rows):', df.shape)
print('Species counts:')
print(df['species'].value_counts())
df.head()

### Peek at the data

Run this to see how two of the body measurements separate the three species. Notice that Gentoo penguins (in particular) tend to stand apart — that is the structure our SVM will exploit.

In [ ]:
colors = {'Adelie': 'tab:blue', 'Chinstrap': 'tab:orange', 'Gentoo': 'tab:green'}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for sp, c in colors.items():
    sub = df[df['species'] == sp]
    axes[0].scatter(sub['flipper_length_mm'], sub['body_mass_g'],
                    c=c, label=sp, alpha=0.6)
    axes[1].scatter(sub['culmen_length_mm'], sub['culmen_depth_mm'],
                    c=c, label=sp, alpha=0.6)
axes[0].set_xlabel('Flipper length (mm)'); axes[0].set_ylabel('Body mass (g)')
axes[0].set_title('Flipper length vs Body mass'); axes[0].legend()
axes[1].set_xlabel('Culmen length (mm)'); axes[1].set_ylabel('Culmen depth (mm)')
axes[1].set_title('Culmen length vs Culmen depth'); axes[1].legend()
plt.tight_layout()
plt.show()

### Train/test split

Run this cell to build a stratified 75/25 split. We keep the raw (unscaled) arrays for now so we can see the effect of scaling in Step 3.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[feature_cols].values
y = df['species'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print('Train:', X_train.shape, '  Test:', X_test.shape)

## Step 2: A Linear SVM Classifier ("Is it a Gentoo?")

In the textbook, the first SVM answered a yes/no question about iris (*"is it Iris virginica?"*) with `LinearSVC`. We'll do the same, but our question is *"is this a **Gentoo**?"*

**Your turn!** Build boolean targets, then train a `LinearSVC` inside a scaling pipeline (exactly as the textbook does).

**Hint (from the textbook):**
```python
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

svm_clf = make_pipeline(StandardScaler(),
                        LinearSVC(C=1, dual=True, random_state=42))
svm_clf.fit(X_train, y_train_gentoo)
```

In [ ]:
# TODO: Create boolean targets y_train_gentoo and y_test_gentoo
#       (True when the species is 'Gentoo').
# y_train_gentoo = ...
# y_test_gentoo  = ...

# TODO: Build a make_pipeline(StandardScaler(), LinearSVC(...)) and fit it
#       on X_train / y_train_gentoo.
# Your code here:



# TODO: Predict whether the first three test penguins are Gentoo,
#       and print the prediction next to the true label.


**Your turn!** Report the **test accuracy** of your Gentoo classifier with `accuracy_score`.

**Hint:**
```python
from sklearn.metrics import accuracy_score
accuracy_score(y_test_gentoo, svm_clf.predict(X_test))
```

In [ ]:
# TODO: Print the test accuracy of the Gentoo classifier.
# Your code here:




## Step 3: Why Feature Scaling Matters for SVMs

The textbook stresses that SVMs are **sensitive to feature scales** — a feature with a large range (like `body_mass_g`, in the thousands) will dominate one with a small range (like `culmen_depth_mm`, around 15–20) unless you scale.

**Your turn!** Train two `LinearSVC` classifiers on the Gentoo task: one **without** scaling (fit directly on `X_train`) and one **with** `StandardScaler` (the pipeline from Step 2). Print both test accuracies side by side.

**Hint:**
```python
unscaled = LinearSVC(C=1, dual=True, random_state=42).fit(X_train, y_train_gentoo)
scaled = make_pipeline(StandardScaler(),
                       LinearSVC(C=1, dual=True, random_state=42)).fit(X_train, y_train_gentoo)
```

In [ ]:
# TODO: Fit an unscaled LinearSVC and a scaled (pipeline) LinearSVC on the
#       Gentoo task. Print both test accuracies.
# Your code here:




**Question:** How much did scaling change the accuracy? In one or two sentences, explain *why* SVMs are so sensitive to feature scale (think about what "maximizing the margin" means when one axis is measured in grams and another in millimeters).

*Your answer:*



## Step 4: The Soft-Margin Trade-off (the `C` hyperparameter)

The textbook shows that `C` controls the **soft margin**: a small `C` allows more margin violations (a wider, more regularized margin that may underfit), while a large `C` tries hard to classify every training point correctly (a narrow margin that may overfit).

**Your turn!** Using a scaled `SVC(kernel='linear', C=...)` pipeline, loop over `C` in `[0.01, 0.1, 1, 10, 100]` and print the test accuracy for each.

**Hint:**
```python
from sklearn.svm import SVC
for C in [0.01, 0.1, 1, 10, 100]:
    clf = make_pipeline(StandardScaler(), SVC(kernel='linear', C=C))
    clf.fit(X_train, y_train_gentoo)
    ...
```

In [ ]:
# TODO: Loop over several C values for a scaled linear SVC on the Gentoo task.
#       Print C and the test accuracy for each.
# Your code here:




**Question:** Which `C` values (if any) underfit, and does accuracy keep improving as `C` grows? Relate what you see to the bias/variance idea: a small `C` is a *more regularized* model. (Gentoo is easy to separate, so the effect may be subtle — note it anyway.)

*Your answer:*



## Step 5: Nonlinear Classification with Kernels

Not every problem is linearly separable. The textbook introduces the **polynomial kernel** and the **RBF (Gaussian) kernel** to draw curved decision boundaries, demonstrated on the two-moons dataset.

### 5a: The two-moons dataset

**Your turn!** Generate `make_moons`, then compare a **linear** kernel SVM against an **RBF** kernel SVM on it. Print both training accuracies. (The moons are *not* linearly separable, so the linear kernel should clearly lose.)

**Hint (from the textbook):**
```python
from sklearn.datasets import make_moons
X_moons, y_moons = make_moons(n_samples=200, noise=0.2, random_state=42)

linear = make_pipeline(StandardScaler(), SVC(kernel='linear', C=1))
rbf = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=5, gamma='scale'))
```

In [ ]:
# TODO: Generate make_moons; fit a linear-kernel and an RBF-kernel SVC;
#       print both training accuracies (use .score(X_moons, y_moons)).
# Your code here:




### 5b: See the boundaries

Run this provided helper to *visualize* the linear vs RBF decision boundaries on the moons. (Nothing to fill in — just run it and look. It refits fresh models so it works even if you named yours differently above.)

In [ ]:
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC

X_moons, y_moons = make_moons(n_samples=200, noise=0.2, random_state=42)

def plot_decision_regions(model, X, y, ax, title):
    model.fit(X, y)
    x0min, x0max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    x1min, x1max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x0min, x0max, 300),
                         np.linspace(x1min, x1max, 300))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap='coolwarm')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='k', s=25)
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_decision_regions(make_pipeline(StandardScaler(), SVC(kernel='linear', C=1)),
                      X_moons, y_moons, axes[0], 'Linear kernel')
plot_decision_regions(make_pipeline(StandardScaler(), SVC(kernel='rbf', C=5, gamma='scale')),
                      X_moons, y_moons, axes[1], 'RBF kernel')
plt.tight_layout()
plt.show()

### 5c: The RBF `gamma` hyperparameter

The RBF kernel has a `gamma` parameter. The textbook shows that a **large `gamma`** makes the decision boundary wiggly and tight around individual points (risk of overfitting), while a **small `gamma`** makes it smoother.

**Your turn!** On the moons data, fit an RBF `SVC` for `gamma` in `[0.1, 1, 10, 100]` (keep `C=5`) and print the **training accuracy** for each. Watch it climb toward 1.0 as `gamma` grows.

**Hint:**
```python
for g in [0.1, 1, 10, 100]:
    clf = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=5, gamma=g))
    clf.fit(X_moons, y_moons)
```

In [ ]:
# TODO: Loop over gamma values for an RBF SVC on the moons; print train accuracy.
# Your code here:




**Question:** As `gamma` increases, the training accuracy rises toward 100%. Why is a training accuracy of 100% *not* automatically a good thing? What is `gamma=100` likely doing to the decision boundary?

*Your answer:*



## Step 6: Multiclass SVM on All Three Species

Now classify all **three** penguin species at once. `SVC` handles multiclass automatically (one-vs-one under the hood).

**Your turn!** Train a scaled RBF `SVC` on the full `y_train` (all three species) and report the test accuracy. Then compare it to a scaled **linear** kernel on the same task.

**Hint:**
```python
rbf = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=5, gamma='scale'))
rbf.fit(X_train, y_train)          # y_train has 3 classes
```

In [ ]:
# TODO: Train a scaled RBF SVC and a scaled linear SVC on the 3-class target.
#       Print both test accuracies.
# Your code here:




**Your turn!** Display the confusion matrix for your best multiclass model so you can see which species (if any) get mixed up.

**Hint:**
```python
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test)
plt.show()
```

In [ ]:
# TODO: Show the confusion matrix for your best multiclass SVM.
# Your code here:




**Question:** Which two species are most likely to be confused with each other, and does that match what you saw in the Step 1 scatter plots? (Hint: think about which two species overlap most in body measurements.)

*Your answer:*



## Reflection Questions

Answer each question in the cell below it (1-3 sentences each).

**Q1:** You saw scaling change the linear SVM's accuracy in Step 3. Why does a `StandardScaler` matter far more for an SVM than it would for, say, a decision tree?

*Your answer:*



**Q2:** The `C` hyperparameter and the RBF `gamma` hyperparameter both influence overfitting. In one sentence each, describe what happens when `C` is very large and when `gamma` is very large.

*Your answer:*



**Q3:** The two-moons data needed a nonlinear kernel, but the penguins were separable with a linear one. How could you *tell* — before trying every kernel — that penguins might be linearly separable? (Think about what the Step 1 scatter plots showed.)

*Your answer:*



**Q4:** The RBF kernel implicitly maps the data into a very high-dimensional space without ever computing the coordinates there (the "kernel trick"). Name one practical advantage of *not* having to compute that mapping explicitly.

*Your answer:*

